In [1]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

In [2]:
from mppi import InputFiles as I, Calculators as C, Datasets as D
from mppi.Datasets import PostProcessing as PP

# yambo_nl simulations of bulk LiF: delta, sine and frequency mixing

This notebook builds and runs the `yambo_nl` simulations used to extract the equilibrium (non-linear) susceptibilities
of LiF. The analysis of the results (extraction of the $\chi$ and reconstruction of $\delta\chi^{neq}$ for the
comparison with the RT transient absorption) is performed in `NL-Chi_Analysis.ipynb`.

We use the SAVE folder computed by D. Sangalli in

_/work/sangalli/simulations/LiF/yambo/e80_kx6_pbe_sr/kx8_nb100/NoTr_E100_

(kx8 grid, 100 bands, time reversal removed for a field along 100), copied in `kx8_nb100_NoTr_E100/`.

Three kinds of simulations are performed:

- a single delta pulse (linear response);
- a set of monochromatic sine-shaped probes in the 10-20 eV range;
- frequency mixing: the same probes with a sine-shaped pump at 1.55 eV.

All the runs are appended with `skip=True`: if the results are already present in the run folder they are not recomputed.

## Ground state: inversion symmetry of the k sampling

LiF is centrosymmetric, so all the even order susceptibilities must vanish. The SAVE of D. Sangalli (fixsym, NoTr)
uses only the 8 spatial symmetries that leave invariant a field along x (point group $4mm$, without inversion) and no
time reversal (the setup report `r_setup` gives `Inversion symmetry: no`, `K-space Time-reversal: no`). So, for most of
the k points of the BZ, the wavefunctions at $\mathbf{k}$ and $-\mathbf{k}$ are not related by a symmetry used by yambo:
they come from independent pw.x calculations. If the k grid were not closed under $\mathbf{k}\to-\mathbf{k}$, or if the
states at $\mathbf{k}$ and $-\mathbf{k}$ were not equivalent, the sampling would break the inversion symmetry of the
system. We check:

1. that the BZ grid obtained by unfolding the IBZ points with the symmetries contains $-\mathbf{k}$ for every
   $\mathbf{k}$;
2. that $E_n(\mathbf{k}) = E_n(-\mathbf{k})$ for all the bands;
3. that the gauge and rotation invariant sums $\sum_{cv}|\mathbf{p}_{cv}|^2$ (valence bands 3-5, conduction band 6)
   and $\sum_{nm}|\mathbf{p}_{nm}|^2$ (bands 1-6) are equal at the IBZ points that represent $\mathbf{k}$ and
   $-\mathbf{k}$. The dipoles are computed by `yambo_nl` at the beginning of each run; we read the ones of the delta
   run (the database is the same for all the runs).

The lattice, the symmetries, the k points and the band energies are read with `YamboDftParser`, which also performs
the expansion of the IBZ to the full BZ (`expand_IBZ_kpoints`) and finds the index of the point at $-\mathbf{k}$
(`get_minus_k_indexes`). The dipoles are read with `YamboDipolesParser`.

In [3]:
from mppi import Parsers as P
from mppi.Utilities.Constants import HaToeV
import numpy as np
import os

save_dir = os.path.join('kx8_nb100_NoTr_E100','SAVE')
dft = P.YamboDftParser(os.path.join(save_dir,'ns.db1'))
dft.get_info()
print('number of symmetries:',len(dft.syms),' time reversal:',dft.time_reversal)

Parse file : kx8_nb100_NoTr_E100/SAVE/ns.db1
YamboDftParser variables structure
number of k points 100
number of bands 100
spin degeneration 1
number of symmetries: 8  time reversal: False


In [ ]:
# 1. expansion of the IBZ to the full BZ and closure of the grid under k -> -k
kbz = dft.expand_IBZ_kpoints()        # BZ points in crystal coordinates
minus = dft.get_minus_k_indexes()     # index of the BZ point at -k (-1 if it is not in the grid)
ibz = dft.ibz_index                   # index of the IBZ point of each BZ point
print('IBZ points in crystal coordinates (first 4):',kbz[np.searchsorted(ibz,range(4))].round(4).tolist())
print('BZ points whose -k is not in the grid: %d'%np.sum(minus<0))
print('BZ points for which k and -k come from different IBZ points: %d'%np.sum(ibz!=ibz[minus]))

Number of symmetries used: 8 (time reversal: False)
100 IBZ k points expanded to 512 BZ k points


IBZ points in crystal coordinates (first 4): [[0.0, 0.0, 0.0], [0.0, 0.0, 0.125], [0.0, 0.0, 0.25], [0.0, 0.0, 0.375]]
BZ points whose -k is not in the grid: 0
BZ points for which k and -k come from different IBZ points: 434


In [ ]:
# 2. band energies at k and -k
evals = dft.evals*HaToeV   # (k, band)
print('max |E_n(k)-E_n(-k)| (all the %d bands): %.2e eV'%(dft.nbands,np.max(np.abs(evals[ibz]-evals[ibz[minus]]))))

# 3. gauge and rotation invariant sums of the momentum dipoles at the IBZ representatives of k and -k
dip = P.YamboDipolesParser(os.path.join('kx8_nb100_NoTr_E100','lresponse-bands_3-6-delta','ndb.dipoles'))
dip_p = dip.dip_p[...,0] + 1j*dip.dip_p[...,1]    # (k, band, band, direction), bands 1-6
val, cond = [2,3,4], [5]                          # bands 3-5 and 6 (0-based)
s_cv = np.array([np.sum(np.abs(p[np.ix_(cond,val)])**2) for p in dip_p])
s_all = np.array([np.sum(np.abs(p)**2) for p in dip_p])
print('max rel. diff. of sum|p_cv|^2 between k and -k: %.2e'%np.max(np.abs(s_cv[ibz]-s_cv[ibz[minus]])/s_cv[ibz]))
print('max rel. diff. of sum|p_nm|^2 between k and -k: %.2e'%np.max(np.abs(s_all[ibz]-s_all[ibz[minus]])/s_all[ibz]))

max |E_n(k)-E_n(-k)| (all the 100 bands): 0.00e+00 eV
Parse file : kx8_nb100_NoTr_E100/lresponse-bands_3-6-delta/ndb.dipoles
Spin dipoles not found in the ndb.dipoles
max rel. diff. of sum|p_cv|^2 between k and -k: 2.34e-15
max rel. diff. of sum|p_nm|^2 between k and -k: 6.50e-16


**Results.** The grid is the full $8\times8\times8$ $\Gamma$-centered grid (512 points in the BZ) and it is closed under
$\mathbf{k}\to-\mathbf{k}$. For most of the BZ points $\mathbf{k}$ and $-\mathbf{k}$ come from different IBZ points,
but the band energies coincide exactly and the dipole sums agree to machine precision: the states at $\mathbf{k}$ and
$-\mathbf{k}$ are equivalent. The strings used for the Berry phase polarization are built on the same grid, so they are
inversion symmetric as well (the coarse 8-point discretization gives an error of order $\Delta k^2$, which is even under
inversion). The k sampling of the ground state does not break the inversion symmetry of LiF.

## RT simulation with yambo_nl in the IP approximation

The parameters are taken from the folder

_/work/sangalli/simulations/LiF/yambo/e80_kx6_pbe_sr/kx8_nb100/NoTr_E100/sh6.0+str0.05_probes/tdberry_

and in particular from `td-ip-ber_3.0as_inv_3-12b_delta1.0fs_deph0.1fs_p_NL`, reducing the bands to 3-6.

In [4]:
# RunRules for ismhpc
# The product of ntasks_per_node*cpus_per_task is equal to 32. 
# Many mpi are needed for better performances
nodes = 1
ntasks_per_node = 32 #16
cpus_per_task = 1 #2
omp_num_threads = 1 #2
module = '/home/dalessandro/module_script/yambo_module'

ntasks = nodes*ntasks_per_node

rr = C.RunRules(scheduler='slurm',partition='all12h', #'all12h',
                memory='125000',
                nodes=nodes,ntasks_per_node=ntasks_per_node,
                cpus_per_task=cpus_per_task,
                omp_num_threads=omp_num_threads,pre_processing=module)
code = C.YamboCalculator(rr,executable='yambo_nl',activate_BeeOND=True)
#code.global_options()

Initialize a Yambo calculator with scheduler slurm


In [4]:
run_dir = 'kx8_nb100_NoTr_E100/'

### Linear response within the Berry phase approach (delta pulse)

We compute the linear response of the system using a single delta shaped pulse. The linear $\chi$ extracted from this
run is compared with the one computed from the sine-shaped pulses below.

The time step is 0.01 fs as in the sine runs: with the INVINT (Crank-Nicolson) integrator a transition energy E evolves as $E_{eff} = (2\hbar/dt)\arctan(E\,dt/2\hbar)$, so a step of 0.05 fs red shifts the 10-20 eV spectrum by ~2 eV (17.4 eV -> 15.4 eV); with 0.01 fs the shift is ~0.1 eV.

In [7]:
NLBands = [3,6]
GfnQP_E = [6.,1.05,1.05]

Field1_Int = 1.e3 
Field1_Dir = [1.,0.,0.]
Field1_kind = 'DELTA'
Field1_Tstart = 0.1 # fs
NLDamping = 0.0 # eV
NLtime, NLstep = 100, 0.01  # fs (0.05 fs is too large with INVINT in the 10-20 eV range: energies red shifted by ~2 eV)
IOtime = [1.,5.,0.005] #fs

inp = I.YamboInput(args='yambo_nl -u p',folder=run_dir)
inp.set_array_variables(NLBands=NLBands,GfnQP_E=GfnQP_E)
inp.set_array_variables(units='eV',NLDamping=NLDamping)
inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int)
inp.set_array_variables(Field1_Dir=Field1_Dir)
inp.set_array_variables(units='fs',Field1_Tstart=Field1_Tstart,NLtime=NLtime,NLstep=NLstep)
inp.set_scalar_variables(Field1_kind=Field1_kind)
inp.set_array_variables(units='fs',IOtime=IOtime)

study = D.Dataset(num_tasks=1,run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
idd = 'lresponse-bands_3-6-delta'
study.append_run(id=idd,input=inp,runner=code,jobname=[idd],skip=True)

Initialize a Dataset with 1 parallel tasks


In [8]:
results = study.run()

Run the selection [0] with the parallel task_groups [[0]] 

Run the task [0] 
delete job_out script: kx8_nb100_NoTr_E100/job_lresponse-bands_3-6-delta.out
delete folder: kx8_nb100_NoTr_E100/lresponse-bands_3-6-delta
run command: mpirun -np 32 yambo_nl -F lresponse-bands_3-6-delta.in -J "lresponse-bands_3-6-delta" -C lresponse-bands_3-6-delta
slurm submit:  cd kx8_nb100_NoTr_E100/ ; sbatch job_lresponse-bands_3-6-delta.sh
computation lresponse-bands_3-6-delta is running...
Submitted batch job 58304
computation lresponse-bands_3-6-delta ended
Task [0] ended 
 


### Sine-shaped probe

We compute the response of the system to a set of monochromatic sine-shaped pulses (one run for each probe frequency)
with an intensity of 1e3 kW/m$^2$. These runs provide the linear $\chi$ in the probe energy range.

In [9]:
NLBands = [3,6]
GfnQP_E = [6.,1.05,1.05]
NLEnRange = [10.0,20.0] # energy range of the probe
NLEnSteps = 201 # number of steps for the loop on frequencies

Field1_Int = 1e3
Field1_Dir = [1.,0.,0.]
Field1_kind = 'SIN'
Field1_Freq = -0.1 # the range is used to set th freq of the pulse
Field1_Tstart = 0.1 # fs
NLDamping = 0.1 # eV
NLtime, NLstep = 200, 0.01  # fs
IOtime = [1.,5.,0.005] #fs

inp = I.YamboInput(args='yambo_nl -u n',folder=run_dir)
inp.set_array_variables(NLBands=NLBands,GfnQP_E=GfnQP_E)
inp.set_array_variables(units='eV',NLDamping=NLDamping,Field1_Freq=Field1_Freq,NLEnRange=NLEnRange)
inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int)
inp.set_array_variables(Field1_Dir=Field1_Dir,NLEnSteps=NLEnSteps)
inp.set_array_variables(units='fs',Field1_Tstart=Field1_Tstart,NLtime=NLtime,NLstep=NLstep)
inp.set_scalar_variables(Field1_kind=Field1_kind)
inp.set_array_variables(units='fs',IOtime=IOtime)

study = D.Dataset(num_tasks=1,run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
idd = 'pulse-E1_1e3-nlenrange_%s-%s-nlensteps_%s-bands_3-6-damp_0.1-sin'%(NLEnRange[0],NLEnRange[1],NLEnSteps)
study.append_run(id=idd,input=inp,runner=code,jobname=[idd],skip=True) #,reformat=False

Initialize a Dataset with 1 parallel tasks


In [10]:
results = study.run()

Run the selection [0] with the parallel task_groups [[0]] 

Run the task [0] 
Skip the run of pulse-E1_1e3-nlenrange_10.0-20.0-nlensteps_201-bands_3-6-damp_0.1-sin
Task [0] ended 
 


### Frequency mixing: probe and pump

The same set of probes is combined with a sine-shaped pump at 1.55 eV with an intensity of 1e6 kW/m$^2$. These runs
provide the mixing susceptibilities $\chi_{(n,m)}$ (n times the probe and m times the pump frequency).

In [12]:
NLBands = [3,6]
GfnQP_E = [6.,1.05,1.05]
NLEnRange = [10.0,20.0] # energy range of the probe
NLEnSteps = 201 

Field1_Freq, Field2_Freq = -0.1, 1.55 # eV Energy of the pump pulse
Field1_Int, Field2_Int = 1.e3, 1.e6 # kW/m^2 (probe, pump)
Field1_Dir, Field2_Dir = [1.,0.,0.], [1.,0.,0.]
Field1_kind, Field2_kind = 'SIN', 'SIN'
Field1_Tstart, Field2_Tstart = 0.1, 0.1 # fs
NLDamping = 0.1 # eV
NLtime, NLstep = 200, 0.01  # fs
IOtime = [1.,5.,0.005] #fs

inp = I.YamboInput(args='yambo_nl -u n',folder=run_dir)
inp.set_array_variables(NLBands=NLBands,GfnQP_E=GfnQP_E,NLEnSteps=NLEnSteps)
inp.set_array_variables(units='eV',NLEnRange=NLEnRange,Field1_Freq=Field1_Freq,Field2_Freq=Field2_Freq,NLDamping=NLDamping)
inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int,Field2_Int=Field2_Int)
inp.set_array_variables(Field1_Dir=Field1_Dir,Field2_Dir=Field2_Dir)
inp.set_array_variables(units='fs',Field1_Tstart=Field1_Tstart,Field2_Tstart=Field2_Tstart,NLtime=NLtime,NLstep=NLstep)
inp.set_scalar_variables(Field1_kind=Field1_kind,Field2_kind=Field2_kind)
inp.set_array_variables(units='fs',IOtime=IOtime)
inp.set_scalar_variables(NL_ROLEs="w.k",NL_CPU='4.8')

study = D.Dataset(num_tasks=1,run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
idd = 'Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_%s-%s-nlensteps_%s-bands_3-6-damp_0.1-sin'%(NLEnRange[0],NLEnRange[1],NLEnSteps)
study.append_run(id=idd,input=inp,runner=code,jobname=[idd],skip=True)

Initialize a Dataset with 1 parallel tasks


In [13]:
results = study.run()

Run the selection [0] with the parallel task_groups [[0]] 

Run the task [0] 
Skip the run of Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_10.0-20.0-nlensteps_201-bands_3-6-damp_0.1-sin
Task [0] ended 
 


### Diagnostic runs for the mixing terms $\chi_{(1,\pm1)}$

The analysis of the frequency mixing runs (`NL-Chi_Analysis.ipynb`) shows that the polarization computed by `yambo_nl`
contains components at $\omega\pm\omega_P$ that are forbidden by the inversion symmetry of LiF, while the second order
response of each field alone vanishes and the k sampling is inversion symmetric (see the beginning of this notebook).
These terms are about 10 times larger than the third order pump induced term and follow the probe resonances
(maximum around 17.5 eV). To identify their origin we repeat the P&p simulation for a few probe frequencies changing
one parameter at a time:

- **reference**: the same parameters of the production run. It is needed because the runs are now performed with the
  rebuilt yambo (Lumen 2.1.0, setup redone), while the production runs were done with Lumen 2.0.0;
- **pump intensity / 4** (2.5e5 kW/m$^2$, i.e. $E_P$ halved);
- **probe intensity x 4** (4e3 kW/m$^2$, i.e. $E_p$ doubled);
- **time step / 2** (`NLstep` = 0.005 fs);
- **Crank-Nicolson integrator** (`NLintegrator` = CRANKNIC). INVINT performs one Cayley step
  $(1-i\,dt\,H(t)/2)/(1+i\,dt\,H(t)/2)$ with the Hamiltonian at the beginning of the step, so it is first order for
  the time (field) and state (Berry coupling) dependent part of $H$. CRANKNIC estimates the bands at $t+dt/2$ with a
  half INVINT step, recomputes $H(t+dt/2)$ and performs the Cayley step with the midpoint Hamiltonian (second order,
  twice the cost). For the static part of $H$ the two schemes have the same phase error, so the shift of the
  resonances is the same and the keys can be compared with the reference run at fixed frequency.

Interpretation:

- a genuine second order (bilinear) response of the simulated system gives $P(\omega\pm\omega_P) \propto E_pE_P$, so
  $\chi_{(1,\pm1)} = P/(E_pE_P)$ does not change in the two intensity runs; any other scaling (e.g. a term that does
  not depend on $E_p$ or that goes as $E_P^2$) points to a numerical artifact of a different nature;
- if $\chi_{(1,\pm1)}$ changes sizeably when the time step is halved, the origin is in the time integration (INVINT
  with the two fields), while the physical keys (linear and third order) must be stable.
- the integrator cannot break the inversion symmetry if all the terms of the dynamics respect it, so we expect the
  same $\chi_{(1,\pm1)}$ with CRANKNIC; a sizeable change would point to the time integration of the state dependent
  coupling with the fields.

We use 5 probe frequencies around the 17.5 eV resonance, chosen to coincide with the indices 149-153 of the production
grid (`NLEnRange` = [10,20] eV, `NLEnSteps` = 201, step 10/201 eV), so that the results can be compared point by point:
with `NLEnRange` = $[a, a+5\delta]$ and `NLEnSteps` = 5 yambo uses the frequencies $a + i\delta$, $i=0,\dots,4$.
These are short runs (5 frequencies instead of 201), submitted on the `all12h` partition as the production runs.

In [8]:
# RunRules for the short diagnostic runs (partition all12h, as the production runs: it runs on all the nodes)
rr_debug = C.RunRules(scheduler='slurm',partition='all12h',
                      memory='125000',
                      nodes=1,ntasks_per_node=32,
                      cpus_per_task=1,omp_num_threads=1,
                      pre_processing='/home/dalessandro/module_script/yambo_module')
code_debug = C.YamboCalculator(rr_debug,executable='yambo_nl',activate_BeeOND=True)

# 5 probe frequencies around 17.5 eV, equal to the indices 149-153 of the production grid
delta_E = 10./201
i_first, n_freqs = 149, 5
NLEnRange_diag = [round(10.+i_first*delta_E,5), round(10.+(i_first+n_freqs)*delta_E,5)]
print('NLEnRange:',NLEnRange_diag,' probe frequencies (eV):',[round(NLEnRange_diag[0]+i*delta_E,4) for i in range(n_freqs)])

Initialize a Yambo calculator with scheduler slurm
NLEnRange: [17.41294, 17.66169]  probe frequencies (eV): [17.4129, 17.4627, 17.5124, 17.5622, 17.6119]


In [9]:
run_dir = 'kx8_nb100_NoTr_E100/'

In [10]:
def build_Pp_input(Field1_Int=1.e3, Field2_Int=1.e6, NLstep=0.01, NLEnRange=NLEnRange_diag, NLEnSteps=n_freqs, NLintegrator='INVINT', folder=None):
    # P&p input with the same parameters of the production run, apart from the arguments
    if folder is None: folder = run_dir
    inp = I.YamboInput(args='yambo_nl -u n',folder=folder)
    inp.set_array_variables(NLBands=[3,6],GfnQP_E=[6.,1.05,1.05],NLEnSteps=NLEnSteps)
    inp.set_array_variables(units='eV',NLEnRange=NLEnRange,Field1_Freq=-0.1,Field2_Freq=1.55,NLDamping=0.1)
    inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int,Field2_Int=Field2_Int)
    inp.set_array_variables(Field1_Dir=[1.,0.,0.],Field2_Dir=[1.,0.,0.])
    inp.set_array_variables(units='fs',Field1_Tstart=0.1,Field2_Tstart=0.1,NLtime=200,NLstep=NLstep)
    inp.set_scalar_variables(Field1_kind='SIN',Field2_kind='SIN',NLintegrator=NLintegrator)
    inp.set_array_variables(units='fs',IOtime=[1.,5.,0.005])
    inp.set_scalar_variables(NL_ROLEs="w.k",NL_CPU='4.8')
    return inp

# (Field1_Int, Field2_Int, NLstep, NLintegrator) of the diagnostic runs
diag_runs = {'reference'   : (1.e3, 1.e6, 0.01, 'INVINT'),
             'pump_int/4'  : (1.e3, 2.5e5, 0.01, 'INVINT'),
             'probe_int*4' : (4.e3, 1.e6, 0.01, 'INVINT'),
             'nlstep/2'    : (1.e3, 1.e6, 0.005, 'INVINT'),
             'cranknic'    : (1.e3, 1.e6, 0.01, 'CRANKNIC')}
integrator_tag = {'INVINT':'', 'CRANKNIC':'-CN'}   # the runs with INVINT (default) have no tag in the id

fmt_int = lambda x : ('%.2g'%x).replace('e+0','e')   # 1000 -> 1e3, 250000 -> 2.5e5

study = D.Dataset(num_tasks=len(diag_runs),run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
diag_ids = {}
for name,(I1,I2,dt,integrator) in diag_runs.items():
    idd = 'Pp-Ep_%s-EP_%s-PE_1.55-nlenrange_%s-%s-nlensteps_%s-bands_3-6-damp_0.1-sin-nlstep_%s%s'%(
          fmt_int(I1),fmt_int(I2),NLEnRange_diag[0],NLEnRange_diag[1],n_freqs,dt,integrator_tag[integrator])
    # yambo truncates the job string at 100 characters: in the ids of the pump_int/4 (101 characters) and cranknic
    # (102 characters) runs the energy range is written with 3 decimals (the frequencies of the runs are not changed).
    # The ids of the other runs are not changed, since they have already been computed
    if len(idd) > 100: idd = idd.replace("nlenrange_%s-%s"%tuple(NLEnRange_diag),"nlenrange_%.3f-%.3f"%tuple(NLEnRange_diag))
    diag_ids[name] = idd
    study.append_run(id=idd,input=build_Pp_input(Field1_Int=I1,Field2_Int=I2,NLstep=dt,NLintegrator=integrator),runner=code_debug,jobname=[idd],skip=True)
diag_ids

Initialize a Dataset with 5 parallel tasks


{'reference': 'Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01',
 'pump_int/4': 'Pp-Ep_1e3-EP_2.5e5-PE_1.55-nlenrange_17.413-17.662-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01',
 'probe_int*4': 'Pp-Ep_4e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01',
 'nlstep/2': 'Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.005',
 'cranknic': 'Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.413-17.662-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01-CN'}

In [11]:
results = study.run()

Run the selection [0, 1, 2, 3, 4] with the parallel task_groups [[0, 1, 2, 3, 4]] 

Run the task [0, 1, 2, 3, 4] 
Skip the run of Skip the run ofPp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01 
Pp-Ep_1e3-EP_2.5e5-PE_1.55-nlenrange_17.413-17.662-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01Skip the run of
 Skip the run of Pp-Ep_4e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01
Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.005
Skip the run of Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.413-17.662-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01-CN
Task [0, 1, 2, 3, 4] ended 
 


### Diagnostic run on the kx12 grid

The diagnostic runs show that the $(1,\pm1)$ terms are a genuine bilinear response of the simulated system and that
they do not depend on the time integrator. The remaining candidate is the spatial discretization, in particular the
coupling with the field in the Berry phase approach, computed from the overlaps between neighbouring k points on the
strings (8 points per string on the kx8 grid). We repeat the reference run (same fields, `NLstep` = 0.01 fs, INVINT,
the same 5 probe frequencies) on the $12\times12\times12$ grid: if the $(1,\pm1)$ terms come from the k discretization,
their size relative to the linear response must decrease ($\Delta k$ is reduced by a factor 2/3).

The SAVE is the one of D. Sangalli (`/work/sangalli/simulations/LiF/yambo/e80_kx6_pbe_sr/kx12_nb100/NoTr_E100/SAVE`):
same lattice, 100 bands, the same 8 symmetries without inversion and time reversal of the kx8 SAVE, 294 IBZ points
(1728 in the BZ). It is copied in `kx12_nb100_NoTr_E100/SAVE` without the setup databases (`ndb.gops`, `ndb.kindx`,
`ndb.kpts`) written by yambo 5.2, that give NaN overlaps with the current build (Lumen 2.1.0): the setup has been
performed again by running `yambo` without arguments in `kx12_nb100_NoTr_E100` (with the yambo module loaded).

In [12]:
run_dir_kx12 = 'kx12_nb100_NoTr_E100'

In [14]:
study_kx12 = D.Dataset(num_tasks=1,run_dir=run_dir_kx12,verbose=True)
study_kx12.set_postprocessing_function(PP.yambo_parse_data)
I1,I2,dt,integrator = diag_runs['reference']
idd = 'Pp-Ep_%s-EP_%s-PE_1.55-nlenrange_%s-%s-nlensteps_%s-bands_3-6-damp_0.1-sin-nlstep_%s'%(
      fmt_int(I1),fmt_int(I2),NLEnRange_diag[0],NLEnRange_diag[1],n_freqs,dt)
print(len(idd),idd)
study_kx12.append_run(id=idd,input=build_Pp_input(Field1_Int=I1,Field2_Int=I2,NLstep=dt,NLintegrator=integrator,folder=run_dir_kx12),
                      runner=code_debug,jobname=[idd],skip=True)

Initialize a Dataset with 1 parallel tasks
99 Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01


In [15]:
results_kx12 = study_kx12.run()

Run the selection [0] with the parallel task_groups [[0]] 

Run the task [0] 
run command: mpirun -np 32 yambo_nl -F Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01.in -J "Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01" -C Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01
slurm submit:  cd kx12_nb100_NoTr_E100 ; sbatch job_Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01.sh
computation Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01 is running...
Submitted batch job 58941
computation Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_17.41294-17.66169-nlensteps_5-bands_3-6-damp_0.1-sin-nlstep_0.01 ended
Run performed in 28m-29s
Task [0] ended 
 


## Convergence of the linear response with the k grid

The comparison of the reference P&p run on the kx8 and kx12 grids shows that $|\chi^{(1)}|$ changes by a factor 2
around 17.5 eV: the linear response is not converged with the k sampling on the kx8 grid in the 10-20 eV window. We
compute the linear response on the kx12 and kx16 grids with a delta pulse, with the same parameters of the kx8 delta
run (bands 3-6, `NLstep` = 0.01 fs, 100 fs, no damping), which gives the whole spectrum with a single run.

The SAVEs are the ones of D. Sangalli (`.../e80_kx6_pbe_sr/kx12_nb100/NoTr_E100/SAVE` and
`.../e80_kx6_pbe_sr/kx16_nb100/NoTr_E100/SAVE`): same lattice, 100 bands and the same 8 symmetries (no inversion and
time reversal) of the kx8 SAVE, with 294 and 648 IBZ points (1728 and 4096 in the BZ). They are copied without the
setup databases of yambo 5.2, and the setup has been performed again by running `yambo` without arguments in each
folder. The runs use 2 nodes (64 MPI tasks; with a single field yambo distributes all the tasks on the k points) and are
collected in a single dataset with `num_tasks=1`, so they are performed one after the other.

In [16]:
# RunRules with 2 nodes (64 MPI tasks) on the all12h partition
rr_2nodes = C.RunRules(scheduler='slurm',partition='all12h',
                       memory='125000',
                       nodes=2,ntasks_per_node=32,
                       cpus_per_task=1,omp_num_threads=1,
                       pre_processing='/home/dalessandro/module_script/yambo_module')
code_2nodes = C.YamboCalculator(rr_2nodes,executable='yambo_nl',activate_BeeOND=True)

def build_delta_input(folder):
    # delta pulse input with the same parameters of the kx8 delta run
    inp = I.YamboInput(args='yambo_nl -u p',folder=folder)
    inp.set_array_variables(NLBands=[3,6],GfnQP_E=[6.,1.05,1.05])
    inp.set_array_variables(units='eV',NLDamping=0.0)
    inp.set_array_variables(units='kWLm2',Field1_Int=1.e3)
    inp.set_array_variables(Field1_Dir=[1.,0.,0.])
    inp.set_array_variables(units='fs',Field1_Tstart=0.1,NLtime=100,NLstep=0.01)
    inp.set_scalar_variables(Field1_kind='DELTA')
    inp.set_array_variables(units='fs',IOtime=[1.,5.,0.005])
    return inp

run_dirs_k = {'kx12':'kx12_nb100_NoTr_E100', 'kx16':'kx16_nb100_NoTr_E100'}
# a single dataset with num_tasks=1: the two runs (2 nodes each) are performed one after the other. Each run is
# performed in its own folder (run_dir keyword of append_run) with the same name of the kx8 delta run
study_delta_k = D.Dataset(num_tasks=1,run_dir=run_dirs_k['kx12'],verbose=True)
study_delta_k.set_postprocessing_function(PP.yambo_parse_data)
name = 'lresponse-bands_3-6-delta'
for grid,folder in run_dirs_k.items():
    study_delta_k.append_run(id='delta-%s'%grid,input=build_delta_input(folder),runner=code_2nodes,
                             run_dir=folder,name=name,jobname=[name],skip=True)
study_delta_k.ids

Initialize a Yambo calculator with scheduler slurm
Initialize a Dataset with 1 parallel tasks


['delta-kx12', 'delta-kx16']

In [17]:
results_delta_k = study_delta_k.run()

Run the selection [0, 1] with the parallel task_groups [[0], [1]] 

Run the task [0] 
run command: mpirun -np 64 yambo_nl -F lresponse-bands_3-6-delta.in -J "lresponse-bands_3-6-delta" -C lresponse-bands_3-6-delta
slurm submit:  cd kx12_nb100_NoTr_E100 ; sbatch job_lresponse-bands_3-6-delta.sh
computation lresponse-bands_3-6-delta is running...
computation lresponse-bands_3-6-delta ended
Run performed in 32m-58s
Task [0] ended 
 
Run the task [1] 
run command: mpirun -np 64 yambo_nl -F lresponse-bands_3-6-delta.in -J "lresponse-bands_3-6-delta" -C lresponse-bands_3-6-delta
slurm submit:  cd kx16_nb100_NoTr_E100 ; sbatch job_lresponse-bands_3-6-delta.sh
computation lresponse-bands_3-6-delta is running...
computation lresponse-bands_3-6-delta ended
Run performed in 56m-24s
Task [1] ended 
 


## Sine and frequency mixing runs with a damping of 0.3 eV

With a damping of 0.1 eV the structures of the spectrum above the gap are not converged with the k grid, while with a
larger broadening the different grids agree (see `NL-Chi_Analysis.ipynb`). The sine and frequency mixing runs are
repeated with a damping of 0.3 eV, which can be further increased a posteriori by the Lorentzian broadening on the
probe frequency grid (reliable up to $\Delta\simeq$ 0.3 eV, so a total broadening of $\simeq$ 0.6 eV as in the RT
spectra of D. Sangalli). With respect to the production runs:

- `NLDamping` = 0.3 eV: the dephasing time of the harmonic analysis (12/damp) is 26 fs instead of 79 fs, so the
  simulation time is reduced to `NLtime` = 100 fs (the fit window of the frequency mixing needs $\simeq$ 21 fs);
- the probe window is 10-25 eV, which covers the whole absorption band of LiF (up to $\sim$ 23 eV) and reduces the
  edge effects of the broadening in the 10-20 eV region;
- the frequency step is $\omega_P/16$ = 0.096875 eV, so that the probe frequencies shifted by $\pm\omega_P$ and
  $\pm2\omega_P$ (needed in the formula of the transient absorption) fall exactly on the grid: yambo uses the
  frequencies `NLEnRange[0]` + i (`NLEnRange[1]`-`NLEnRange[0]`)/`NLEnSteps`, i = 0,...,`NLEnSteps`-1, so
  `NLEnRange` = [10, 25.015625] eV and `NLEnSteps` = 155 give the frequencies 10.000-24.919 eV;
- 2 nodes (64 MPI tasks) with `NL_ROLEs` = "w.k" and `NL_CPU` = "8.8": 8 groups of frequencies with 8 tasks on the k
  points each.

The other parameters (bands 3-6, scissor, intensities 1e3 kW/m$^2$ for the probe and 1e6 kW/m$^2$ for the pump at
1.55 eV, `NLstep` = 0.01 fs, INVINT) are the ones of the production runs.

The comparison of the delta runs on the kx8, kx12 and kx16 grids shows that with a broadening of 0.3 eV the kx12
grid is converged within a few per cent, while the kx8 one still has an artifact at 17.5 eV. The runs are performed
on both the kx8 grid (comparison with the production runs with damping 0.1 eV) and the kx12 grid (converged results).
The four runs are collected in a single dataset with `num_tasks=1`, so they are performed one after the other (each
run on 2 nodes, in the folder of its grid).

In [ ]:
grids_eta = ['kx8','kx12']   # k grids of the runs
run_dirs_eta = {'kx8':'kx8_nb100_NoTr_E100', 'kx12':'kx12_nb100_NoTr_E100', 'kx16':'kx16_nb100_NoTr_E100'}

omega_P_eV = 1.55
NLEnSteps_eta = 155
dE_eta = omega_P_eV/16                                     # frequency step (eV)
NLEnRange_eta = [10.0, 10.0+NLEnSteps_eta*dE_eta]          # [10, 25.015625] eV
NLDamping_eta, NLtime_eta = 0.3, 100                       # eV, fs
print('NLEnRange:',NLEnRange_eta,' step %.6f eV, probe frequencies %.3f-%.3f eV'%(
      dE_eta,NLEnRange_eta[0],NLEnRange_eta[0]+(NLEnSteps_eta-1)*dE_eta))

def build_nl_input(folder, with_pump):
    # sine (probe only) or frequency mixing (probe + pump) input with damping 0.3 eV on the 10-25 eV window
    inp = I.YamboInput(args='yambo_nl -u n',folder=folder)
    inp.set_array_variables(NLBands=[3,6],GfnQP_E=[6.,1.05,1.05],NLEnSteps=NLEnSteps_eta)
    inp.set_array_variables(units='eV',NLEnRange=NLEnRange_eta,NLDamping=NLDamping_eta,Field1_Freq=-0.1)
    inp.set_array_variables(units='kWLm2',Field1_Int=1.e3)
    inp.set_array_variables(Field1_Dir=[1.,0.,0.])
    inp.set_array_variables(units='fs',Field1_Tstart=0.1,NLtime=NLtime_eta,NLstep=0.01)
    inp.set_scalar_variables(Field1_kind='SIN')
    if with_pump:
        inp.set_array_variables(units='eV',Field2_Freq=omega_P_eV)
        inp.set_array_variables(units='kWLm2',Field2_Int=1.e6)
        inp.set_array_variables(Field2_Dir=[1.,0.,0.])
        inp.set_array_variables(units='fs',Field2_Tstart=0.1)
        inp.set_scalar_variables(Field2_kind='SIN')
    inp.set_array_variables(units='fs',IOtime=[1.,5.,0.005])
    inp.set_scalar_variables(NL_ROLEs="w.k",NL_CPU='8.8')
    return inp

range_tag = 'nlenrange_%s-%.3f-nlensteps_%s'%(NLEnRange_eta[0],NLEnRange_eta[1],NLEnSteps_eta)
ids_eta = {'sine' : 'pulse-E1_1e3-%s-bands_3-6-damp_%s-sin-nltime_%s'%(range_tag,NLDamping_eta,NLtime_eta),
           'Pp'   : 'Pp-Ep_1e3-EP_1e6-PE_1.55-%s-bands_3-6-damp_%s-sin-nltime_%s'%(range_tag,NLDamping_eta,NLtime_eta)}
study_eta = D.Dataset(num_tasks=1,run_dir=run_dirs_eta[grids_eta[0]],verbose=True)
study_eta.set_postprocessing_function(PP.yambo_parse_data)
for grid in grids_eta:
    folder = run_dirs_eta[grid]
    for kind,idd in ids_eta.items():
        assert len(idd) <= 100, idd   # yambo truncates the job string at 100 characters
        # the id of the dataset contains the grid, the name of the run (input, output and database folder) does not
        study_eta.append_run(id='%s-%s'%(grid,kind),input=build_nl_input(folder,with_pump=(kind=='Pp')),runner=code_2nodes,
                             run_dir=folder,name=idd,jobname=[idd],skip=True)
study_eta.ids

In [ ]:
results_eta = study_eta.run()